# Piloto encoder vs. lineal — Fase 2 (BCCh)

Corre en **Colab con GPU** la comparación pareada BETO vs. arquitectura lineal W+C sobre el mismo split agrupado por reunión, usando el script [`scripts/piloto_encoder_dev.py`](https://github.com/joako0o/FASE_2/blob/arena/01a0c4da-fase-2/scripts/piloto_encoder_dev.py) clonado directo del repo público (rama `arena/01a0c4da-fase-2`).

**Antes de correr todo:** menú *Entorno de ejecución → Cambiar tipo de entorno → GPU (T4)* y guardar.

**Duración estimada (T4):** corrida rápida `--semillas 2` ≈ 25–35 min; corrida completa `--grid --semillas 5` ≈ 2–3 h. Empieza por la rápida para validar el entorno.

No requiere subir nada: el clon trae el CSV de entrenamiento y el notebook verifica su SHA-256 contra el `manifest.json` del propio repo antes de entrenar.


In [ ]:
!nvidia-smi -L || echo '⚠️ NO HAY GPU: menú Entorno de ejecución → Cambiar tipo de entorno → GPU'


In [ ]:
!pip install -q transformers accelerate

import transformers, torch
print('torch', torch.__version__, '| transformers', transformers.__version__, '| cuda:', torch.cuda.is_available())


## 1. Clonar el repo y verificar integridad


In [ ]:
![ -d FASE_2 ] || git clone --depth 1 --branch arena/01a0c4da-fase-2 https://github.com/joako0o/FASE_2.git
import hashlib, json
man = json.load(open('FASE_2/data/manifest.json', encoding='utf-8'))['sha256']
ruta = 'FASE_2/data/entrenamiento_wc600.csv'
clave = ruta.split('FASE_2/', 1)[1]  # el manifest usa rutas relativas a la raiz del repo
sha = hashlib.sha256(open(ruta, 'rb').read()).hexdigest()
assert sha == man[clave], f'CLON ALTERADO: {sha[:16]}… != {man[clave][:16]}…'
print(f'Clone OK y entrenamiento integro (sha {sha[:16]}…)')


## 2. Correr el piloto

**Recomendado:** primero la corrida rápida (línea corta), y si todo bien, la completa con `--grid` (tuning simétrico con GroupKFold(3) interno) y 5 semillas.

El script entrena BETO en dos etapas (A relevancia, B H/D/N) que espejan el sistema formal, y el brazo lineal en el mismo split; reporta media ± desv por semilla y bootstrap **pareado** de la diferencia remuestreando reuniones completas. La evaluación ciega permanece cerrada.


In [ ]:
# Corrida rápida (~25-35 min): descomenta la línea siguiente y comenta la larga

# !python FASE_2/scripts/piloto_encoder_dev.py --semillas 2



# Corrida del paper (~2-3 h):

!python FASE_2/scripts/piloto_encoder_dev.py --grid --semillas 5


## 3. Resumen y descarga del resultado


In [ ]:
from google.colab import files

import json

r = json.load(open('FASE_2/resultados/robustez_supervisada/piloto_encoder_dev.json', encoding='utf-8'))

print('LINEAL      :', {k: round(r['lineal'][k], 4) for k in ('macro_f1','f1_hd','f1_d')})

md_ = r['encoder_media_desv']

print('ENCODER     :', {k: f"{md_[k]['media']:.4f}±{md_[k]['desv']:.4f}" for k in md_})

print()

print('Δ pareado (encoder − lineal), IC95 bootstrap por semilla:')

for k, v in r['bootstrap_pareado_delta_encoder_menos_lineal'].items():

    ics = [f"[{d['ic95'][0]:.3f}; {d['ic95'][1]:.3f}]" for d in v['detalle_por_semilla']]

    print(f"  {k:10s} semillas con IC sin 0: {v['semillas_ic_excluye_cero']}/5 | {ics}")

print()

print('Lectura: si los IC de macro_f1/F1-HD contienen 0 en todas las semillas -> titular de COSTO (el lineal no es distinguible del encoder), no de victoria.')

files.download('FASE_2/resultados/robustez_supervisada/piloto_encoder_dev.json')


## Qué devolver al repo

1. Pega en el chat la salida de la celda 2 (los IC del Δ): con eso se fija el titular de la sección B del paper.
2. Sube el `piloto_encoder_dev.json` descargado (o pégalo en el chat) y se integra a `resultados/robustez_supervisada/` con su pin en `resultados/manifest.json`.

Si Colab free corta la sesión en la corrida larga, los parciales se pierden: deja la corrida completa en una sesión que puedas tener quieta ~3 h.
